In [1]:
import sys
from pathlib import Path

import ee
import pandas as pd

# tests/backtests.ipynb → RozviDrought project root
PROJECT_ROOT = Path.cwd().resolve()

# If notebook is launched from tests/, move one level up.
if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

EE_PROJECT = "august-analyze"

try:
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine initialized with project: {EE_PROJECT}")
except Exception as exc:
    print("Earth Engine not initialized. Running authentication...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine authenticated and initialized with project: {EE_PROJECT}")

print("Project root:", PROJECT_ROOT)

Earth Engine initialized with project: august-analyze
Project root: C:\Projects\Infer RozviDrought\RozviDrought


In [2]:
# Cell 2: Load Zimbabwe administrative polygons from Earth Engine

ADMIN_LEVEL = 2
GAUL_ASSET = f"FAO/GAUL/2015/level{ADMIN_LEVEL}"

admin_fc = (
    ee.FeatureCollection(GAUL_ASSET)
    .filter(ee.Filter.eq("ADM0_NAME", "Zimbabwe"))
)

admin_count = admin_fc.size().getInfo()

first_admin = admin_fc.first().toDictionary().getInfo()

print(f"Loaded Zimbabwe ADM{ADMIN_LEVEL} polygons from:", GAUL_ASSET)
print("Admin polygon count:", admin_count)
print("Example properties:")
first_admin

Loaded Zimbabwe ADM2 polygons from: FAO/GAUL/2015/level2
Admin polygon count: 62
Example properties:


{'ADM0_CODE': 271,
 'ADM0_NAME': 'Zimbabwe',
 'ADM1_CODE': 3436,
 'ADM1_NAME': 'Harare',
 'ADM2_CODE': 68807,
 'ADM2_NAME': 'Chitungwiza',
 'DISP_AREA': 'NO',
 'EXP2_YEAR': 3000,
 'STATUS': 'Member State',
 'STR2_YEAR': 2006,
 'Shape_Area': 0.00407260399874,
 'Shape_Leng': 0.34197890796}

In [3]:
# Cell 3: Pull Zimbabwe ADM2 polygons from Earth Engine into local records

admin_features = admin_fc.getInfo()["features"]

admin_records = []

for feature in admin_features:
    props = feature["properties"]
    geom = feature["geometry"]

    admin_records.append({
        "adm0_name": props.get("ADM0_NAME"),
        "adm1_name": props.get("ADM1_NAME"),
        "adm2_name": props.get("ADM2_NAME"),
        "adm2_code": props.get("ADM2_CODE"),
        "geometry": geom,
    })

admin_df = pd.DataFrame(admin_records)

print("Admin records loaded:", len(admin_df))
print(admin_df[["adm1_name", "adm2_name", "adm2_code"]].head())

Admin records loaded: 62
             adm1_name    adm2_name  adm2_code
0               Harare  Chitungwiza      68807
1               Harare       Harare      68809
2  Mashonaland Central       Guruve      68808
3  Mashonaland Central        Mbire      68811
4           Manicaland       Makoni      33056


In [ ]:
# Cell 4: Convert ADM2 GeoJSON geometries to Shapely geometries

from shapely.geometry import shape

admin_df["shapely_geometry"] = admin_df["geometry"].apply(shape)

print("Converted ADM2 geometries to Shapely.")
print(admin_df[["adm1_name", "adm2_name", "adm2_code"]].head())
print("Example geometry type:", admin_df.loc[0, "shapely_geometry"].geom_type)